In [ ]:
import os
os.chdir("..")

In [ ]:
from prsal_repl import PrsalReg
import bambi as bmb
import arviz as az
import seaborn as sns
import arviz_plots as azp

import matplotlib.pyplot as plt
from patsy import dmatrix
from sklearn.linear_model import Ridge
azp.style.use("arviz-variat")


prsal = PrsalReg()

In [ ]:
df = prsal.make_pr_dataset()
df.columns

In [ ]:
sns.pairplot(data[[
       "k_index",
       "zip_employment",
       'commute_car',
       'own_children6', 
       'own_children17', 
       "food_stamp",
       'with_social_security']]);

In [ ]:
from pandas.plotting import autocorrelation_plot

# Autocorrelation plots for each variable
for col in [
       "k_index",
       "zip_employment",
       'commute_car',
       'own_children6', 
       'own_children17', 
       "food_stamp",
       'with_social_security']:
    plt.figure()
    autocorrelation_plot(data[[
       "k_index",
       "zip_employment",
       'commute_car',
       'own_children6', 
       'own_children17', 
       "food_stamp",
       'with_social_security']])
    plt.title(f'Autocorrelation for {col}')
    plt.show()

In [ ]:
data.describe()

In [ ]:
data = df[[
       "zipcode", 
       "year", 
       "qtr", 
       "k_index",
       "zip_employment",
       'commute_car',
       'own_children6', 
       'own_children17', 
       "food_stamp",
       'with_social_security', 
       'wq_employment', 
       'wq_k_index',
       'wk6_employment', 
       'wk6_k_index', 
       'wd30_employment',
       'wd30_k_index',
       "geometry"]]
data["time"] =  data["year"]*10 + data["qtr"]
data["zipcode"] = data["zipcode"].astype("category")
data["time"] = data["time"].astype("category")
data.head()

In [ ]:

# Queens model 
model = bmb.Model(
    """ 
    zip_employment ~ k_index 
                        + k_index
                        + commute_car
                        + own_children6 
                        + own_children17 
                        + food_stamp
                        + with_social_security
                        + wq_employment 
                        + wq_k_index 
                        + C(zipcode) 
                        + C(time)
       """,
    data=data,
    family="gaussian"
)
resutls_wq = model.fit(
    draws=1000,
    seed=787,
    tune=1000,
    target_accept=0.9
)

# Knn30 mils
model = bmb.Model(
    """ 
    zip_employment ~ k_index 
                        + commute_car
                        + own_children6 
                        + own_children17 
                        + food_stamp
                        + with_social_security
                        + wd30_employment 
                        + wd30_k_index 
                        + C(zipcode) 
                        + C(time)
       """,
    data=data,
    family="gaussian"
)
resutls_wd30 = model.fit(
    draws=1000,
    seed=787,
    tune=1000,
    target_accept=0.9
)

# Knn 6 neighbors
model = bmb.Model(
    """ 
    zip_employment ~ k_index 
                        + commute_car
                        + own_children6 
                        + own_children17 
                        + food_stamp
                        + with_social_security
                        + wk6_employment 
                        + wk6_k_index 
                        + C(zipcode) 
                        + C(time)
       """,
    data=data,
    family="gaussian"
)
resutls_wk6 = model.fit(
    draws=1000,
    seed=787,
    tune=1000,
    target_accept=0.9
)

In [ ]:
az.summary(resutls_wq, var_names=[
    "k_index",
    "sigma", 
    "Intercept",
    'commute_car',
    'own_children6', 
    'own_children17', 
    "food_stamp",
    'with_social_security', 
    'wq_employment', 
    'wq_k_index',
])

In [ ]:
az.plot_trace(resutls_wq, compact=False, var_names=[
    "k_index",
    "sigma", 
    "Intercept",
    'commute_car',
    'own_children6', 
    'own_children17', 
    "food_stamp",
    'with_social_security', 
    'wq_employment', 
    'wq_k_index',
]);

In [ ]:
az.summary(resutls_wk6,var_names=[
    "k_index",
    "sigma", 
    "Intercept",
    'commute_car',
    'own_children6', 
    'own_children17', 
    "food_stamp",
    'with_social_security', 
    'wk6_employment', 
    'wk6_k_index', 
])

In [ ]:
az.plot_trace(resutls_wk6, compact=False, var_names=[
    "k_index",
    "sigma", 
    "Intercept",
    'commute_car',
    'own_children6', 
    'own_children17', 
    "food_stamp",
    'with_social_security', 
    'wk6_employment', 
    'wk6_k_index', 
]);

In [ ]:
az.summary(resutls_wd30, var_names=[
    "k_index",
    "sigma", 
    "Intercept",
    'commute_car',
    'own_children6', 
    'own_children17', 
    "food_stamp",
    'with_social_security', 
    'wd30_employment',
    'wd30_k_index'
    ])

In [ ]:
az.plot_trace(resutls_wd30, compact=False, var_names=[
    "k_index",
    "sigma", 
    "Intercept",
    'commute_car',
    'own_children6', 
    'own_children17', 
    "food_stamp",
    'with_social_security', 
    'wd30_employment',
    'wd30_k_index'
    ]);

In [ ]:
# Project to a planar CRS
gdf_proj = df.to_crs(epsg=5070)

# Extract coordinates
gdf_proj["x"] = gdf_proj.geometry.centroid.x
gdf_proj["y"] = gdf_proj.geometry.centroid.y

In [ ]:

formula = "k_index + te(cr(x, df=6), cr(y, df=6), constraints='center')"

# Build the design matrix
design = dmatrix(formula, gdf_proj, return_type="dataframe")

# Column names (for reference)
column_names = design.design_info.column_names

In [ ]:
# Ridge regression as penalized regression
model = Ridge(alpha=1e-3)  # small ridge penalty
model.fit(design, gdf_proj["zip_employment"])

In [ ]:
coef = model.coef_
coef_dict = dict(zip(column_names, coef))

# Intercept is separate
intercept = model.intercept_

In [ ]:
gdf_proj["y_pred"] = model.predict(design)

In [ ]:
# Predictions
y_pred = model.predict(design)

# Add to GeoDataFrame for plotting
gdf_proj["y_pred"] = y_pred

In [ ]:
import numpy as np

# Use projected coordinates (meters) for proper distances
gdf_plot = gdf_proj.copy()

# Centroids of each polygon
gdf_plot["x"] = gdf_plot.geometry.centroid.x
gdf_plot["y"] = gdf_plot.geometry.centroid.y

# Predicted values from Ridge / tensor regression
gdf_plot["z"] = gdf_plot["y_pred"]

In [ ]:
from scipy.interpolate import griddata

# Grid for surface
grid_x, grid_y = np.meshgrid(
    np.linspace(gdf_plot["x"].min(), gdf_plot["x"].max(), 200),
    np.linspace(gdf_plot["y"].min(), gdf_plot["y"].max(), 200)
)

# Interpolate predicted z values onto the grid
grid_z = griddata(
    points=(gdf_plot["x"], gdf_plot["y"]),
    values=gdf_plot["z"],
    xi=(grid_x, grid_y),
    method='cubic'
)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Add the interpolated surface
fig.add_trace(go.Surface(
    x=grid_x,
    y=grid_y,
    z=grid_z,
    colorscale='Viridis',
    colorbar=dict(title='Predicted Employment'),
    showscale=True,
    opacity=0.9
))

# Optional: overlay zip centroids as points
fig.add_trace(go.Scatter3d(
    x=gdf_plot["x"],
    y=gdf_plot["y"],
    z=gdf_plot["z"],
    mode='markers',
    marker=dict(size=3, color='red'),
    name='Zip Centroids'
))

# Layout
fig.update_layout(
    title="3D Interactive Spatial Surface of Zip Employment",
    scene=dict(
        xaxis_title="X (meters)",
        yaxis_title="Y (meters)",
        zaxis_title="Predicted Employment"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:
gdf_zip = prsal.zips_goem().to_crs(epsg=5070)
# gdf_zip = gdf_zip[~gdf_zip["zipcode"].isin(["00820","00850","00851"])]
gdf_zip.plot()

In [ ]:
import geopandas as gpd
from shapely.geometry import MultiPolygon, Polygon

gdf_zip = gdf_zip[~gdf_zip["zipcode"].isin(["00820","00850","00851"])]

# Explode multi-part geometries into single polygons
gdf_exploded = gdf_zip.explode(ignore_index=True)

# Optional: keep only polygons that intersect others (connected)
connected_mask = gdf_exploded.geometry.apply(
    lambda geom: gdf_exploded.geometry.intersects(geom).sum() > 1
)
gdf_connected = gdf_exploded[connected_mask].copy()

# Or, if you just want to keep all polygons that are part of a MultiPolygon
# but remove isolated ones entirely, you can skip the mask and just work with gdf_exploded

In [ ]:
import numpy as np
import plotly.graph_objects as go

fig = go.Figure()

# --- Add the predicted surface ---
fig.add_trace(go.Surface(
    x=grid_x,
    y=grid_y,
    z=grid_z,
    colorscale='Viridis',
    colorbar=dict(title='Predicted Employment'),
    showscale=True,
    opacity=0.9
))

# --- Overlay ZIP geometries ---
for poly in gdf_zip.geometry:
    if poly.geom_type == "Polygon":
        x, y = poly.exterior.xy
        x = list(x)  # convert array.array to list
        y = list(y)
        z = [0] * len(x)  # set at z=0
        fig.add_trace(go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode='lines',
            line=dict(color='black', width=2),
            name='ZIP Boundary'
        ))
    elif poly.geom_type == "MultiPolygon":
        for subpoly in poly.geoms:
            x, y = subpoly.exterior.xy
            x = list(x)
            y = list(y)
            z = [0] * len(x)
            fig.add_trace(go.Scatter3d(
                x=x,
                y=y,
                z=z,
                mode='lines',
                line=dict(color='black', width=2),
                name='ZIP Boundary'
            ))

# --- Optional: overlay centroids as points ---
gdf_zip_proj = gdf_zip.to_crs(epsg=5070)  # ensure projected coordinates
gdf_zip_proj["x"] = gdf_zip_proj.geometry.centroid.x
gdf_zip_proj["y"] = gdf_zip_proj.geometry.centroid.y
gdf_zip_proj["z"] = 0  # reference at surface base

fig.add_trace(go.Scatter3d(
    x=gdf_zip_proj["x"],
    y=gdf_zip_proj["y"],
    z=gdf_zip_proj["z"],
    mode='markers',
    marker=dict(size=3, color='red'),
    name='Zip Centroids'
))

# --- Layout ---
fig.update_layout(
    title="3D Interactive Spatial Surface with ZIP Boundaries",
    scene=dict(
        xaxis_title="X (meters)",
        yaxis_title="Y (meters)",
        zaxis_title="Predicted Employment"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:
az.plot_dist(resutls_wq)